# Exploratory Data Analysis — Multilingual Health QA
Languages: Akan (Aka_Gha), Amharic (Amh_Eth), Luganda (Lug_Uga), Swahili (Swa_Ken), English (Eng_Uga / Eng_Gha / Eng_Eth / Eng_Ken)

In [ ]:
!pip install -q matplotlib seaborn pandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

train = pd.read_csv('../Train.csv')
val   = pd.read_csv('../Val.csv')
test  = pd.read_csv('../Test.csv')

print('Train:', train.shape, '| Val:', val.shape, '| Test:', test.shape)
train.head(3)

## 1. Subset (Language-Country) Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, df, title in zip(axes, [train, val, test], ['Train', 'Val', 'Test']):
    counts = df['subset'].value_counts()
    ax.bar(counts.index, counts.values, color=sns.color_palette('muted', len(counts)))
    ax.set_title(f'{title} Subset Distribution')
    ax.set_xlabel('Subset')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    for p, v in zip(ax.patches, counts.values):
        ax.text(p.get_x() + p.get_width()/2, p.get_height() + 30, str(v), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('subset_distribution.png', bbox_inches='tight')
plt.show()
print(train['subset'].value_counts())

## 2. Question (Input) Length Distribution

In [ ]:
train['input_len'] = train['input'].fillna('').apply(lambda x: len(x.split()))
train['output_len'] = train['output'].fillna('').apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train['input_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Input Question Length (words)')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(train['input_len'].median(), color='red', linestyle='--', label=f'Median={train["input_len"].median()}')
axes[0].legend()

axes[1].hist(train['output_len'], bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Output Answer Length (words)')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(train['output_len'].median(), color='red', linestyle='--', label=f'Median={train["output_len"].median()}')
axes[1].legend()
plt.tight_layout()
plt.savefig('length_distributions.png', bbox_inches='tight')
plt.show()

print(train[['input_len','output_len']].describe().round(1))

## 3. Answer Length by Subset

In [ ]:
plt.figure(figsize=(12, 5))
order = train.groupby('subset')['output_len'].median().sort_values(ascending=False).index
sns.boxplot(data=train, x='subset', y='output_len', order=order, palette='muted')
plt.title('Answer Length Distribution by Subset')
plt.xlabel('Subset')
plt.ylabel('Answer Word Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('answer_len_by_subset.png', bbox_inches='tight')
plt.show()

## 4. Missing / Empty Values Check

In [ ]:
for name, df in [('Train', train), ('Val', val), ('Test', test)]:
    print(f'--- {name} ---')
    print(df.isnull().sum())
    if 'input' in df.columns:
        print('Empty inputs:', (df['input'].str.strip() == '').sum())
    if 'output' in df.columns:
        print('Empty outputs:', (df['output'].str.strip() == '').sum())
    print()

## 5. Train / Val / Test Split Comparison

In [ ]:
splits = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test'],
    'Rows': [len(train), len(val), len(test)],
})

plt.figure(figsize=(6, 4))
bars = plt.bar(splits['Split'], splits['Rows'], color=['steelblue', 'darkorange', 'green'])
for bar, v in zip(bars, splits['Rows']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, str(v), ha='center')
plt.title('Dataset Split Sizes')
plt.ylabel('Number of Samples')
plt.tight_layout()
plt.savefig('split_sizes.png', bbox_inches='tight')
plt.show()

## 6. Sample Examples Per Language

In [ ]:
for subset in train['subset'].unique():
    row = train[train['subset'] == subset].iloc[0]
    print(f'=== {subset} ===')
    print('Q:', row['input'][:200])
    print('A:', row['output'][:200])
    print()

## 7. Input vs Output Length Correlation

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(train['input_len'], train['output_len'], alpha=0.15, s=5, color='steelblue')
plt.xlabel('Input Length (words)')
plt.ylabel('Output Length (words)')
plt.title('Input vs Output Length Correlation')
corr = train['input_len'].corr(train['output_len'])
plt.text(5, 450, f'Pearson r = {corr:.3f}', fontsize=12, color='red')
plt.tight_layout()
plt.savefig('input_output_correlation.png', bbox_inches='tight')
plt.show()